# 3. Retrieve, check the evidence, and try again

The agent can search two collections: LangGraph documentation and LangChain documentation. We keep the original tutorial's corpus so the framework port can face the same questions.

Run `uv run llamaindex-rag setup --with-reranker` first. GoodMem stores, chunks and embeds the documents. LlamaIndex runs the workflow; the shared integration supplies its standard retriever.

The explicit loop is: **decide → retrieve → grade → draft or rewrite → decide**. A relevance check decides whether to draft from useful passages or suggest a better search. The original question and earlier evidence survive every step. Four searches is the hard limit.

In [ ]:
from goodmem_rag.config import Settings, chat_model
from goodmem_rag.retrieval import make_tools

settings = Settings.from_env()
state = settings.state()
model = chat_model()

In [ ]:
from goodmem_rag.agents import AgenticRAGWorkflow

async with settings.async_client() as client:
    tools = make_tools(async_client=client, state=state)
    print([(tool.metadata.name, tool.metadata.description) for tool in tools])
    workflow = AgenticRAGWorkflow(model, tools)
    result = await workflow.run(question="What is a checkpointer used for in LangGraph? Cite the docs.")

print(result.rendered())
print("Steps:", " → ".join(result.steps))
print("Searches:", result.calls)

## Ask a question that requires a dependent lookup

A multi-hop question needs information from one search to choose the next. Merely searching two collections in parallel is not enough for this example.

In [ ]:
from goodmem_rag.evaluation import SEQUENTIAL_QUESTION

async with settings.async_client() as client:
    result = await AgenticRAGWorkflow(model, make_tools(async_client=client, state=state)).run(
        question=SEQUENTIAL_QUESTION)

print(result.rendered())
for call in result.calls:
    print(call["round"], call["name"], call["args"])

Open `goodmem_rag/agents.py` to follow the typed events and workflow steps. Change `max_searches` to see the budget affect the route. Source links come from stored metadata and remain available on `result.nodes`, independently of the model's answer.